# 07 _ MLflow Experiment Tracking

**Project:** Corporacion Favorita Store Sales Forecasting  
**Project type:** Data Science / MLOps Portfolio Project  
**Notebook:** `notebooks/07_mlflow_tracking.ipynb`

---

## 1.Purpose

This notebook adds a local MLflow tracking layer to the forecasting workflow.

The goal is to register existing model results, parameters, metrics, artifacts, and model-selection evidence without retraining models.


# 2.Notebook Objective

The objective of this notebook is to make experimentation traceable and reproducible for the next MLOps phase.

The notebook will:

- Configure local MLflow tracking with a SQLite backend.
- Validate required input artifacts.
- Load existing metrics, configs, and evaluation summaries.
- Register baseline experiments.
- Register the LightGBM experiment.
- Register the Prophet benchmark experiment.
- Register the final model recommendation.
- Save compact MLflow run summaries and reproducibility notes.


# 3.Scope.

## Scope.

This notebook behaves as a post-training tracking layer.

It includes:

- Local MLflow configuration.
- Artifact inventory validation.
- Existing experiment registration.
- Run summary and registered-artifact reports.
- Reproducibility notes and limitations.

It does not include:

- EDA.
- Data cleaning.
- Feature creation.
- Model retraining.
- Hyperparameter tuning.
- API or Docker development.
- Production monitoring.


# 4.Imports and Path Setup

This section imports the required libraries, detects the project root, defines standard project paths, and configures MLflow to use a local SQLite tracking database.


In [ ]:
# Standard library
import json
import os
import platform
import re
import warnings
from datetime import datetime
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Notebook display
from IPython.display import display

# MLflow
try:
    import mlflow
except ImportError as error:
    raise ImportError(
        "MLflow is required for this notebook. " "Install it with: pip install mlflow"
    ) from error

# Optional model flavor support
try:
    import lightgbm as lgb
    import mlflow.lightgbm

    LIGHTGBM_AVAILABLE = True
except ImportError:
    lgb = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

SEED = 42
np.random.seed(SEED)


def find_project_root(
    start_path=None,
    required_dirs=("data", "notebooks", "reports", "models"),
):
    """Find the project root by walking upward from the current directory."""
    current_path = Path.cwd() if start_path is None else Path(start_path)
    current_path = current_path.resolve()

    candidate_paths = [current_path] + list(current_path.parents)

    for candidate_path in candidate_paths:
        if all((candidate_path / directory).exists() for directory in required_dirs):
            return candidate_path

    raise FileNotFoundError(
        "Project root could not be detected. "
        f"Expected a parent directory containing: {required_dirs}. "
        f"Current working directory: {current_path}"
    )


# Project paths
PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
FEATURES_DIR = DATA_DIR / "features"
PREDICTIONS_DIR = DATA_DIR / "predictions"

MODELS_DIR = PROJECT_ROOT / "models"
BASELINES_MODELS_DIR = MODELS_DIR / "baselines"
LIGHTGBM_MODELS_DIR = MODELS_DIR / "lightgbm"
PROPHET_MODELS_DIR = MODELS_DIR / "prophet"

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_TABLES_DIR = REPORTS_DIR / "tables"
REPORTS_FIGURES_DIR = REPORTS_DIR / "figures"
REPORTS_MLFLOW_DIR = REPORTS_DIR / "mlflow"

MLFLOW_TRACKING_DB = PROJECT_ROOT / "mlflow.db"
MLFLOW_EXPERIMENT_NAME = "store_sales_forecasting"
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_TRACKING_DB.as_posix()}"

# Ensure output directories exist
REPORTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_DB.parent.mkdir(parents=True, exist_ok=True)

# Configure MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

setup_summary = pd.DataFrame(
    [
        {"setting": "project_root", "value": str(PROJECT_ROOT)},
        {"setting": "mlflow_tracking_uri", "value": MLFLOW_TRACKING_URI},
        {"setting": "mlflow_experiment_name", "value": MLFLOW_EXPERIMENT_NAME},
        {"setting": "python_version", "value": platform.python_version()},
        {"setting": "pandas_version", "value": pd.__version__},
        {"setting": "numpy_version", "value": np.__version__},
        {"setting": "mlflow_version", "value": mlflow.__version__},
        {"setting": "lightgbm_available", "value": LIGHTGBM_AVAILABLE},
        {"setting": "operating_system", "value": platform.platform()},
    ]
)

display(setup_summary)
print("Section conclusion: local MLflow tracking is configured successfully.")


# 5.Expected artifacts and path inventory

This section defines the artifacts expected from previous notebooks and validates their existence before any MLflow run is created.

Artifacts are split into:

- **required artifacts**, which are needed for a defensible tracking layer;
- **optional artifacts**, which are logged when available but do not block execution.

### Section conclusion

Tracking should only proceed after the minimum experiment evidence is available.


In [ ]:
# Model versions
BEST_BASELINE_MODEL = None
LIGHTGBM_MODEL_VERSION = "lightgbm_v1"
PROPHET_MODEL_VERSION = "prophet_total_sales_v1"

# Forecasting contract
PREDICTION_GRAIN = ["date", "store_nbr", "family"]
TARGET_COLUMN = "sales"
PREDICTION_COLUMN = "prediction"
EXPECTED_FORECAST_HORIZON_DAYS = 16

# Baseline artifacts
BASELINE_VALIDATION_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / "baseline_validation_predictions.parquet"
)
BASELINE_METRICS_PATH = REPORTS_TABLES_DIR / "baseline_metrics.csv"
BASELINE_METRICS_BY_DATE_PATH = REPORTS_TABLES_DIR / "baseline_metrics_by_date.csv"
BASELINE_METRICS_BY_STORE_PATH = REPORTS_TABLES_DIR / "baseline_metrics_by_store.csv"
BASELINE_METRICS_BY_FAMILY_PATH = REPORTS_TABLES_DIR / "baseline_metrics_by_family.csv"
BASELINE_METRICS_BY_PROMOTION_PATH = (
    REPORTS_TABLES_DIR / "baseline_metrics_by_promotion.csv"
)
BASELINE_METRICS_BY_HORIZON_PATH = (
    REPORTS_TABLES_DIR / "baseline_metrics_by_horizon_day.csv"
)
BASELINE_CONFIG_PATH = BASELINES_MODELS_DIR / "baseline_config.json"
BASELINE_LEADERBOARD_PATH = BASELINES_MODELS_DIR / "baseline_leaderboard.csv"

# LightGBM artifacts
LIGHTGBM_MODEL_PATH = LIGHTGBM_MODELS_DIR / f"{LIGHTGBM_MODEL_VERSION}_model.txt"
LIGHTGBM_CONFIG_PATH = LIGHTGBM_MODELS_DIR / f"{LIGHTGBM_MODEL_VERSION}_config.json"
LIGHTGBM_FEATURE_LIST_PATH = (
    LIGHTGBM_MODELS_DIR / f"{LIGHTGBM_MODEL_VERSION}_feature_list.json"
)
LIGHTGBM_EXPERIMENT_SUMMARY_PATH = (
    LIGHTGBM_MODELS_DIR / f"{LIGHTGBM_MODEL_VERSION}_experiment_summary.json"
)
LIGHTGBM_VS_BEST_BASELINE_PATH = (
    LIGHTGBM_MODELS_DIR / f"{LIGHTGBM_MODEL_VERSION}_vs_best_baseline.csv"
)

LIGHTGBM_VALIDATION_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"{LIGHTGBM_MODEL_VERSION}_validation_predictions.parquet"
)
LIGHTGBM_TEST_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"{LIGHTGBM_MODEL_VERSION}_test_predictions.parquet"
)
LIGHTGBM_SUBMISSION_PATH = PREDICTIONS_DIR / f"{LIGHTGBM_MODEL_VERSION}_submission.csv"

LIGHTGBM_METRICS_PATH = REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_metrics.csv"
LIGHTGBM_BASELINE_COMPARISON_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_baseline_comparison.csv"
)
LIGHTGBM_TRAINING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_training_summary.csv"
)
LIGHTGBM_TRAINING_DIAGNOSTICS_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_training_diagnostics.csv"
)
LIGHTGBM_OVERFITTING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_overfitting_summary.csv"
)
LIGHTGBM_LEARNING_CURVE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_learning_curve.csv"
)
LIGHTGBM_FEATURE_IMPORTANCE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_feature_importance.csv"
)
LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_feature_group_importance.csv"
)
LIGHTGBM_METRICS_BY_DATE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_metrics_by_date.csv"
)
LIGHTGBM_METRICS_BY_STORE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_metrics_by_store.csv"
)
LIGHTGBM_METRICS_BY_FAMILY_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_metrics_by_family.csv"
)
LIGHTGBM_METRICS_BY_PROMOTION_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_metrics_by_promotion.csv"
)
LIGHTGBM_METRICS_BY_HORIZON_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_metrics_by_horizon_day.csv"
)
LIGHTGBM_SEGMENT_COMPARISON_BY_DATE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_segment_comparison_by_date.csv"
)
LIGHTGBM_SEGMENT_COMPARISON_BY_HORIZON_PATH = (
    REPORTS_TABLES_DIR
    / f"{LIGHTGBM_MODEL_VERSION}_segment_comparison_by_horizon_day.csv"
)
LIGHTGBM_SEGMENT_COMPARISON_BY_STORE_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_segment_comparison_by_store.csv"
)
LIGHTGBM_SEGMENT_COMPARISON_BY_FAMILY_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_segment_comparison_by_family.csv"
)
LIGHTGBM_SEGMENT_COMPARISON_BY_PROMOTION_PATH = (
    REPORTS_TABLES_DIR / f"{LIGHTGBM_MODEL_VERSION}_segment_comparison_by_promotion.csv"
)

# Prophet artifacts
PROPHET_MODEL_PATH = PROPHET_MODELS_DIR / f"{PROPHET_MODEL_VERSION}_model.json"
PROPHET_CONFIG_PATH = PROPHET_MODELS_DIR / f"{PROPHET_MODEL_VERSION}_config.json"
PROPHET_EXPERIMENT_SUMMARY_PATH = (
    PROPHET_MODELS_DIR / f"{PROPHET_MODEL_VERSION}_experiment_summary.json"
)
PROPHET_VALIDATION_PREDICTIONS_PATH = (
    PREDICTIONS_DIR / f"{PROPHET_MODEL_VERSION}_validation_predictions.parquet"
)
PROPHET_METRICS_PATH = REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_metrics.csv"
PROPHET_DAILY_COMPARISON_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_vs_lightgbm_daily_comparison.csv"
)
PROPHET_AGGREGATE_MODEL_COMPARISON_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_aggregate_model_comparison.csv"
)
PROPHET_VS_LIGHTGBM_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_vs_lightgbm_summary.csv"
)
PROPHET_TRAINING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_training_summary.csv"
)
PROPHET_COMPONENT_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_component_summary.csv"
)
PROPHET_COMPONENT_DETAIL_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_validation_component_detail.csv"
)
PROPHET_WEEKLY_COMPONENT_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / f"{PROPHET_MODEL_VERSION}_weekly_component_summary.csv"
)
PROPHET_FORECAST_FIGURE_PATH = (
    REPORTS_FIGURES_DIR / f"{PROPHET_MODEL_VERSION}_forecast.png"
)
PROPHET_COMPONENTS_FIGURE_PATH = (
    REPORTS_FIGURES_DIR / f"{PROPHET_MODEL_VERSION}_components.png"
)

# Final evaluation artifacts from notebook 06
FINAL_MODEL_LEADERBOARD_PATH = REPORTS_TABLES_DIR / "final_model_leaderboard.csv"
FINAL_GRANULAR_COMPARISON_PATH = (
    REPORTS_TABLES_DIR / "final_granular_model_comparison.csv"
)
FINAL_DAILY_COMPARISON_PATH = (
    REPORTS_TABLES_DIR / "final_daily_aggregate_model_comparison.csv"
)
FINAL_DECISION_MATRIX_PATH = REPORTS_TABLES_DIR / "final_model_decision_matrix.csv"
FINAL_MODEL_RECOMMENDATION_PATH = REPORTS_TABLES_DIR / "final_model_recommendation.csv"
FINAL_ERROR_BY_DATE_PATH = REPORTS_TABLES_DIR / "final_error_by_date.csv"
FINAL_ERROR_BY_HORIZON_PATH = REPORTS_TABLES_DIR / "final_error_by_horizon_day.csv"
FINAL_ERROR_BY_STORE_PATH = REPORTS_TABLES_DIR / "final_error_by_store.csv"
FINAL_ERROR_BY_FAMILY_PATH = REPORTS_TABLES_DIR / "final_error_by_family.csv"
FINAL_ERROR_BY_PROMOTION_PATH = REPORTS_TABLES_DIR / "final_error_by_promotion.csv"
FINAL_ERROR_BY_HOLIDAY_PATH = REPORTS_TABLES_DIR / "final_error_by_holiday.csv"
MLFLOW_ARTIFACTS_RECOMMENDATION_PATH = (
    REPORTS_TABLES_DIR / "mlflow_artifacts_recommendation.csv"
)

FINAL_GLOBAL_METRICS_FIGURE_PATH = (
    REPORTS_FIGURES_DIR / "final_model_global_metrics_comparison.png"
)
FINAL_DAILY_ACTUAL_VS_PREDICTION_FIGURE_PATH = (
    REPORTS_FIGURES_DIR / "final_daily_actual_vs_prediction.png"
)
FINAL_ERROR_BY_DATE_FIGURE_PATH = REPORTS_FIGURES_DIR / "final_error_by_date.png"
FINAL_WORST_STORES_FIGURE_PATH = REPORTS_FIGURES_DIR / "final_worst_stores_by_wape.png"
FINAL_WORST_FAMILIES_FIGURE_PATH = (
    REPORTS_FIGURES_DIR / "final_worst_families_by_wape.png"
)

# Outputs created by this notebook
MLFLOW_ARTIFACT_INVENTORY_PATH = REPORTS_TABLES_DIR / "mlflow_artifact_inventory.csv"
MLFLOW_TRACKING_SCHEMA_PATH = REPORTS_TABLES_DIR / "mlflow_tracking_schema.csv"
MLFLOW_RUN_SUMMARY_PATH = REPORTS_TABLES_DIR / "mlflow_run_summary.csv"
MLFLOW_REGISTERED_ARTIFACTS_PATH = (
    REPORTS_TABLES_DIR / "mlflow_registered_artifacts.csv"
)
MLFLOW_FINAL_MODEL_TRACKING_SUMMARY_PATH = (
    REPORTS_TABLES_DIR / "mlflow_final_model_tracking_summary.csv"
)
MLFLOW_REPRODUCIBILITY_NOTES_PATH = (
    REPORTS_TABLES_DIR / "mlflow_reproducibility_notes.csv"
)
MLFLOW_NOTEBOOK_CLOSURE_PATH = (
    REPORTS_TABLES_DIR / "mlflow_notebook_closure_criteria.csv"
)

required_artifacts = {
    "baseline_metrics": BASELINE_METRICS_PATH,
    "baseline_config": BASELINE_CONFIG_PATH,
    "baseline_validation_predictions": BASELINE_VALIDATION_PREDICTIONS_PATH,
    "lightgbm_model": LIGHTGBM_MODEL_PATH,
    "lightgbm_config": LIGHTGBM_CONFIG_PATH,
    "lightgbm_feature_list": LIGHTGBM_FEATURE_LIST_PATH,
    "lightgbm_metrics": LIGHTGBM_METRICS_PATH,
    "lightgbm_validation_predictions": LIGHTGBM_VALIDATION_PREDICTIONS_PATH,
    "prophet_model": PROPHET_MODEL_PATH,
    "prophet_config": PROPHET_CONFIG_PATH,
    "prophet_metrics": PROPHET_METRICS_PATH,
    "prophet_validation_predictions": PROPHET_VALIDATION_PREDICTIONS_PATH,
    "final_model_leaderboard": FINAL_MODEL_LEADERBOARD_PATH,
    "final_granular_model_comparison": FINAL_GRANULAR_COMPARISON_PATH,
    "final_daily_aggregate_model_comparison": FINAL_DAILY_COMPARISON_PATH,
    "final_model_decision_matrix": FINAL_DECISION_MATRIX_PATH,
    "final_model_recommendation": FINAL_MODEL_RECOMMENDATION_PATH,
}

optional_artifacts = {
    "baseline_leaderboard": BASELINE_LEADERBOARD_PATH,
    "baseline_metrics_by_date": BASELINE_METRICS_BY_DATE_PATH,
    "baseline_metrics_by_store": BASELINE_METRICS_BY_STORE_PATH,
    "baseline_metrics_by_family": BASELINE_METRICS_BY_FAMILY_PATH,
    "baseline_metrics_by_promotion": BASELINE_METRICS_BY_PROMOTION_PATH,
    "baseline_metrics_by_horizon_day": BASELINE_METRICS_BY_HORIZON_PATH,
    "lightgbm_experiment_summary": LIGHTGBM_EXPERIMENT_SUMMARY_PATH,
    "lightgbm_vs_best_baseline": LIGHTGBM_VS_BEST_BASELINE_PATH,
    "lightgbm_test_predictions": LIGHTGBM_TEST_PREDICTIONS_PATH,
    "lightgbm_submission": LIGHTGBM_SUBMISSION_PATH,
    "lightgbm_baseline_comparison": LIGHTGBM_BASELINE_COMPARISON_PATH,
    "lightgbm_training_summary": LIGHTGBM_TRAINING_SUMMARY_PATH,
    "lightgbm_training_diagnostics": LIGHTGBM_TRAINING_DIAGNOSTICS_PATH,
    "lightgbm_overfitting_summary": LIGHTGBM_OVERFITTING_SUMMARY_PATH,
    "lightgbm_learning_curve": LIGHTGBM_LEARNING_CURVE_PATH,
    "lightgbm_feature_importance": LIGHTGBM_FEATURE_IMPORTANCE_PATH,
    "lightgbm_feature_group_importance": LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH,
    "lightgbm_metrics_by_date": LIGHTGBM_METRICS_BY_DATE_PATH,
    "lightgbm_metrics_by_store": LIGHTGBM_METRICS_BY_STORE_PATH,
    "lightgbm_metrics_by_family": LIGHTGBM_METRICS_BY_FAMILY_PATH,
    "lightgbm_metrics_by_promotion": LIGHTGBM_METRICS_BY_PROMOTION_PATH,
    "lightgbm_metrics_by_horizon_day": LIGHTGBM_METRICS_BY_HORIZON_PATH,
    "lightgbm_segment_comparison_by_date": LIGHTGBM_SEGMENT_COMPARISON_BY_DATE_PATH,
    "lightgbm_segment_comparison_by_horizon_day": LIGHTGBM_SEGMENT_COMPARISON_BY_HORIZON_PATH,
    "lightgbm_segment_comparison_by_store": LIGHTGBM_SEGMENT_COMPARISON_BY_STORE_PATH,
    "lightgbm_segment_comparison_by_family": LIGHTGBM_SEGMENT_COMPARISON_BY_FAMILY_PATH,
    "lightgbm_segment_comparison_by_promotion": LIGHTGBM_SEGMENT_COMPARISON_BY_PROMOTION_PATH,
    "prophet_experiment_summary": PROPHET_EXPERIMENT_SUMMARY_PATH,
    "prophet_daily_comparison": PROPHET_DAILY_COMPARISON_PATH,
    "prophet_aggregate_model_comparison": PROPHET_AGGREGATE_MODEL_COMPARISON_PATH,
    "prophet_vs_lightgbm_summary": PROPHET_VS_LIGHTGBM_SUMMARY_PATH,
    "prophet_training_summary": PROPHET_TRAINING_SUMMARY_PATH,
    "prophet_component_summary": PROPHET_COMPONENT_SUMMARY_PATH,
    "prophet_component_detail": PROPHET_COMPONENT_DETAIL_PATH,
    "prophet_weekly_component_summary": PROPHET_WEEKLY_COMPONENT_SUMMARY_PATH,
    "prophet_forecast_figure": PROPHET_FORECAST_FIGURE_PATH,
    "prophet_components_figure": PROPHET_COMPONENTS_FIGURE_PATH,
    "final_error_by_date": FINAL_ERROR_BY_DATE_PATH,
    "final_error_by_horizon_day": FINAL_ERROR_BY_HORIZON_PATH,
    "final_error_by_store": FINAL_ERROR_BY_STORE_PATH,
    "final_error_by_family": FINAL_ERROR_BY_FAMILY_PATH,
    "final_error_by_promotion": FINAL_ERROR_BY_PROMOTION_PATH,
    "final_error_by_holiday": FINAL_ERROR_BY_HOLIDAY_PATH,
    "mlflow_artifacts_recommendation": MLFLOW_ARTIFACTS_RECOMMENDATION_PATH,
    "final_global_metrics_figure": FINAL_GLOBAL_METRICS_FIGURE_PATH,
    "final_daily_actual_vs_prediction_figure": FINAL_DAILY_ACTUAL_VS_PREDICTION_FIGURE_PATH,
    "final_error_by_date_figure": FINAL_ERROR_BY_DATE_FIGURE_PATH,
    "final_worst_stores_figure": FINAL_WORST_STORES_FIGURE_PATH,
    "final_worst_families_figure": FINAL_WORST_FAMILIES_FIGURE_PATH,
}


def file_size_mb(path):
    """Return file size in MB if the file exists."""
    path = Path(path)

    if not path.exists():
        return np.nan

    return round(path.stat().st_size / (1024**2), 4)


def relative_to_project(path):
    """Return a project-relative path when possible."""
    path = Path(path)

    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


artifact_inventory = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "required": True,
            "path": artifact_path,
            "relative_path": relative_to_project(artifact_path),
            "exists": artifact_path.exists(),
            "file_size_mb": file_size_mb(artifact_path),
        }
        for artifact_name, artifact_path in required_artifacts.items()
    ]
    + [
        {
            "artifact": artifact_name,
            "required": False,
            "path": artifact_path,
            "relative_path": relative_to_project(artifact_path),
            "exists": artifact_path.exists(),
            "file_size_mb": file_size_mb(artifact_path),
        }
        for artifact_name, artifact_path in optional_artifacts.items()
    ]
)

artifact_inventory.to_csv(MLFLOW_ARTIFACT_INVENTORY_PATH, index=False)

display(
    artifact_inventory.sort_values(
        ["required", "exists", "artifact"], ascending=[False, True, True]
    )
)

missing_required_artifacts = artifact_inventory.query(
    "required == True and exists == False"
)

if not missing_required_artifacts.empty:
    raise FileNotFoundError(
        "Missing required artifacts for MLflow tracking:\n"
        + "\n".join(
            [
                f"- {row.artifact}: {row.path}"
                for row in missing_required_artifacts.itertuples(index=False)
            ]
        )
    )

print("Section conclusion: all required artifacts are available for MLflow tracking.")


# 6.Helper functions

This section defines reusable utilities for:

- reading existing artifacts;
- flattening nested configuration dictionaries;
- safely logging MLflow parameters and metrics;
- logging artifacts only when they exist;
- extracting compact segment-level summary metrics.

These helpers keep the tracking cells short and reduce repeated code.

### Section conclusion

The notebook will use shared logging helpers to keep each MLflow run consistent.


In [ ]:
logged_artifact_records = []


def read_csv_required(path, artifact_name):
    """Read a required CSV artifact."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required CSV artifact not found: {artifact_name} | {path}"
        )

    df = pd.read_csv(path)
    print(f"Loaded {artifact_name}: {df.shape[0]:,} rows, {df.shape[1]:,} columns")
    return df


def read_csv_optional(path, artifact_name):
    """Read an optional CSV artifact when available."""
    path = Path(path)

    if not path.exists():
        print(f"Optional CSV artifact not found: {artifact_name}")
        return None

    return read_csv_required(path, artifact_name)


def read_json_required(path, artifact_name):
    """Read a required JSON artifact."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Required JSON artifact not found: {artifact_name} | {path}"
        )

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    print(f"Loaded {artifact_name}: {path.name}")
    return data


def read_json_optional(path, artifact_name):
    """Read an optional JSON artifact when available."""
    path = Path(path)

    if not path.exists():
        print(f"Optional JSON artifact not found: {artifact_name}")
        return None

    return read_json_required(path, artifact_name)


def make_json_serializable(value):
    """Convert common Python, NumPy, pandas and pathlib objects into JSON-serializable values."""
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        return float(value)

    if isinstance(value, (np.ndarray,)):
        return value.tolist()

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if isinstance(value, dict):
        return {str(key): make_json_serializable(item) for key, item in value.items()}

    if isinstance(value, (list, tuple)):
        return [make_json_serializable(item) for item in value]

    return value


def save_json(data, path):
    """Save a dictionary as formatted JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as file:
        json.dump(
            make_json_serializable(data),
            file,
            indent=4,
            ensure_ascii=False,
        )


def clean_mlflow_key(value):
    """Convert a string into a conservative MLflow-compatible key."""
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9_./-]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def clean_param_value(value, max_length=500):
    """Convert a parameter value into a compact MLflow-safe representation."""
    value = make_json_serializable(value)

    if isinstance(value, (dict, list, tuple)):
        value = json.dumps(value, ensure_ascii=False, default=str)

    value = str(value)

    if len(value) > max_length:
        value = value[: max_length - 3] + "..."

    return value


def flatten_dict(data, parent_key="", sep=".", max_depth=3, current_depth=0):
    """Flatten a nested dictionary for MLflow parameter logging."""
    flattened = {}

    if not isinstance(data, dict):
        return flattened

    for key, value in data.items():
        clean_key = clean_mlflow_key(key)
        new_key = f"{parent_key}{sep}{clean_key}" if parent_key else clean_key

        if isinstance(value, dict) and current_depth < max_depth:
            flattened.update(
                flatten_dict(
                    value,
                    parent_key=new_key,
                    sep=sep,
                    max_depth=max_depth,
                    current_depth=current_depth + 1,
                )
            )
        else:
            flattened[new_key] = value

    return flattened


def log_params_safely(params, prefix=None, max_length=500):
    """Log MLflow parameters after cleaning keys and values."""
    if not params:
        return

    for key, value in params.items():
        metric_key = clean_mlflow_key(f"{prefix}.{key}" if prefix else key)
        mlflow.log_param(metric_key, clean_param_value(value, max_length=max_length))


def is_valid_metric_value(value):
    """Return True when a value can be logged as a finite MLflow metric."""
    try:
        value = float(value)
    except (TypeError, ValueError):
        return False

    return np.isfinite(value)


def log_metrics_safely(metrics, prefix=None):
    """Log finite numeric MLflow metrics."""
    if not metrics:
        return

    for key, value in metrics.items():
        if is_valid_metric_value(value):
            metric_key = clean_mlflow_key(f"{prefix}_{key}" if prefix else key)
            mlflow.log_metric(metric_key, float(value))


def row_to_dict(row):
    """Convert a pandas Series or namedtuple-like row to a dictionary."""
    if isinstance(row, pd.Series):
        return row.to_dict()

    if hasattr(row, "_asdict"):
        return row._asdict()

    if isinstance(row, dict):
        return row

    return dict(row)


def extract_common_metrics(row, include_prefix=True):
    """Extract common forecasting metrics from a row-like object."""
    row = row_to_dict(row)

    metric_columns = [
        "rows",
        "validation_rows",
        "rmsle",
        "mae",
        "wape",
        "bias",
        "total_bias_pct",
        "actual_total_sales",
        "predicted_total_sales",
        "train_rmse_log",
        "valid_rmse_log",
        "rmse_log_gap",
        "best_iteration",
        "training_time_seconds",
        "training_time_minutes",
    ]

    metrics = {}

    for column in metric_columns:
        if column in row and is_valid_metric_value(row[column]):
            metric_name = (
                f"valid_{column}"
                if include_prefix
                and column
                not in [
                    "train_rmse_log",
                    "valid_rmse_log",
                    "rmse_log_gap",
                    "best_iteration",
                    "training_time_seconds",
                    "training_time_minutes",
                ]
                else column
            )
            metrics[metric_name] = row[column]

    return metrics


def segment_summary_metrics(df, segment_name):
    """Create compact scalar metrics from a segment-level table."""
    if df is None or df.empty:
        return {}

    metrics = {}

    for metric_col in ["rmsle", "mae", "wape", "bias", "total_bias_pct"]:
        if metric_col in df.columns:
            values = pd.to_numeric(df[metric_col], errors="coerce").dropna()

            if not values.empty:
                metrics[f"{segment_name}_{metric_col}_mean"] = values.mean()
                metrics[f"{segment_name}_{metric_col}_max"] = values.max()
                metrics[f"{segment_name}_{metric_col}_min"] = values.min()

    if "rows" in df.columns:
        rows = pd.to_numeric(df["rows"], errors="coerce").dropna()

        if not rows.empty:
            metrics[f"{segment_name}_segments"] = len(df)
            metrics[f"{segment_name}_max_rows"] = rows.max()

    return metrics


def log_artifact_if_exists(path, artifact_path):
    """Log an artifact to MLflow if it exists and keep a local audit record."""
    path = Path(path)

    if not path.exists():
        return False

    mlflow.log_artifact(str(path), artifact_path=artifact_path)

    active_run = mlflow.active_run()
    run_id = active_run.info.run_id if active_run is not None else None
    run_name = (
        active_run.data.tags.get("mlflow.runName") if active_run is not None else None
    )

    logged_artifact_records.append(
        {
            "run_id": run_id,
            "run_name": run_name,
            "artifact_path": artifact_path,
            "source_path": str(path),
            "relative_source_path": relative_to_project(path),
            "file_size_mb": file_size_mb(path),
            "logged_at": datetime.now().isoformat(timespec="seconds"),
        }
    )

    return True


def log_artifact_group(artifact_paths, artifact_path):
    """Log a list of artifacts into the same MLflow artifact directory."""
    logged_count = 0

    for path in artifact_paths:
        if path is not None and log_artifact_if_exists(path, artifact_path):
            logged_count += 1

    return logged_count


def set_common_tags(model_family, model_role, model_version=None, granularity=None):
    """Apply common project tags to the active MLflow run."""
    tags = {
        "project": "store_sales_forecasting",
        "dataset": "corporacion_favorita",
        "notebook": "07_mlflow_tracking.ipynb",
        "tracking_type": "post_training_tracking",
        "training_status": "not_retrained",
        "validation_strategy": "temporal_holdout_16_days",
        "forecast_horizon_days": str(EXPECTED_FORECAST_HORIZON_DAYS),
        "target_column": TARGET_COLUMN,
        "primary_metric": "rmsle",
        "model_family": model_family,
        "model_role": model_role,
    }

    if model_version is not None:
        tags["model_version"] = model_version

    if granularity is not None:
        tags["granularity"] = granularity

    mlflow.set_tags(tags)


def first_row(df):
    """Return the first row as a Series, or an empty Series when unavailable."""
    if df is None or df.empty:
        return pd.Series(dtype=object)

    return df.iloc[0]


print("Section conclusion: reusable MLflow tracking helpers are ready.")


# 7.Load existing metrics, configs and evaluation outputs

This section loads only summary artifacts required for tracking.

Large prediction files are not loaded into memory because they only need to be registered as MLflow artifacts.

### Section conclusion

The notebook will use saved evidence from previous notebooks instead of recomputing results.


In [ ]:
# Required core artifacts
baseline_metrics = read_csv_required(BASELINE_METRICS_PATH, "baseline_metrics")
baseline_config = read_json_required(BASELINE_CONFIG_PATH, "baseline_config")

lightgbm_metrics = read_csv_required(LIGHTGBM_METRICS_PATH, "lightgbm_metrics")
lightgbm_config = read_json_required(LIGHTGBM_CONFIG_PATH, "lightgbm_config")
lightgbm_feature_list = read_json_required(
    LIGHTGBM_FEATURE_LIST_PATH, "lightgbm_feature_list"
)

prophet_metrics = read_csv_required(PROPHET_METRICS_PATH, "prophet_metrics")
prophet_config = read_json_required(PROPHET_CONFIG_PATH, "prophet_config")

final_model_leaderboard = read_csv_required(
    FINAL_MODEL_LEADERBOARD_PATH,
    "final_model_leaderboard",
)
final_granular_comparison = read_csv_required(
    FINAL_GRANULAR_COMPARISON_PATH,
    "final_granular_model_comparison",
)
final_daily_comparison = read_csv_required(
    FINAL_DAILY_COMPARISON_PATH,
    "final_daily_aggregate_model_comparison",
)
final_decision_matrix = read_csv_required(
    FINAL_DECISION_MATRIX_PATH,
    "final_model_decision_matrix",
)
final_model_recommendation = read_csv_required(
    FINAL_MODEL_RECOMMENDATION_PATH,
    "final_model_recommendation",
)

# Optional tabular artifacts
optional_tables = {
    "baseline_leaderboard": read_csv_optional(
        BASELINE_LEADERBOARD_PATH, "baseline_leaderboard"
    ),
    "baseline_metrics_by_date": read_csv_optional(
        BASELINE_METRICS_BY_DATE_PATH, "baseline_metrics_by_date"
    ),
    "baseline_metrics_by_store": read_csv_optional(
        BASELINE_METRICS_BY_STORE_PATH, "baseline_metrics_by_store"
    ),
    "baseline_metrics_by_family": read_csv_optional(
        BASELINE_METRICS_BY_FAMILY_PATH, "baseline_metrics_by_family"
    ),
    "baseline_metrics_by_promotion": read_csv_optional(
        BASELINE_METRICS_BY_PROMOTION_PATH, "baseline_metrics_by_promotion"
    ),
    "baseline_metrics_by_horizon_day": read_csv_optional(
        BASELINE_METRICS_BY_HORIZON_PATH, "baseline_metrics_by_horizon_day"
    ),
    "lightgbm_baseline_comparison": read_csv_optional(
        LIGHTGBM_BASELINE_COMPARISON_PATH, "lightgbm_baseline_comparison"
    ),
    "lightgbm_vs_best_baseline": read_csv_optional(
        LIGHTGBM_VS_BEST_BASELINE_PATH, "lightgbm_vs_best_baseline"
    ),
    "lightgbm_training_summary": read_csv_optional(
        LIGHTGBM_TRAINING_SUMMARY_PATH, "lightgbm_training_summary"
    ),
    "lightgbm_training_diagnostics": read_csv_optional(
        LIGHTGBM_TRAINING_DIAGNOSTICS_PATH, "lightgbm_training_diagnostics"
    ),
    "lightgbm_overfitting_summary": read_csv_optional(
        LIGHTGBM_OVERFITTING_SUMMARY_PATH, "lightgbm_overfitting_summary"
    ),
    "lightgbm_feature_importance": read_csv_optional(
        LIGHTGBM_FEATURE_IMPORTANCE_PATH, "lightgbm_feature_importance"
    ),
    "lightgbm_feature_group_importance": read_csv_optional(
        LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH, "lightgbm_feature_group_importance"
    ),
    "lightgbm_metrics_by_date": read_csv_optional(
        LIGHTGBM_METRICS_BY_DATE_PATH, "lightgbm_metrics_by_date"
    ),
    "lightgbm_metrics_by_store": read_csv_optional(
        LIGHTGBM_METRICS_BY_STORE_PATH, "lightgbm_metrics_by_store"
    ),
    "lightgbm_metrics_by_family": read_csv_optional(
        LIGHTGBM_METRICS_BY_FAMILY_PATH, "lightgbm_metrics_by_family"
    ),
    "lightgbm_metrics_by_promotion": read_csv_optional(
        LIGHTGBM_METRICS_BY_PROMOTION_PATH, "lightgbm_metrics_by_promotion"
    ),
    "lightgbm_metrics_by_horizon_day": read_csv_optional(
        LIGHTGBM_METRICS_BY_HORIZON_PATH, "lightgbm_metrics_by_horizon_day"
    ),
    "prophet_daily_comparison": read_csv_optional(
        PROPHET_DAILY_COMPARISON_PATH, "prophet_daily_comparison"
    ),
    "prophet_aggregate_model_comparison": read_csv_optional(
        PROPHET_AGGREGATE_MODEL_COMPARISON_PATH, "prophet_aggregate_model_comparison"
    ),
    "prophet_vs_lightgbm_summary": read_csv_optional(
        PROPHET_VS_LIGHTGBM_SUMMARY_PATH, "prophet_vs_lightgbm_summary"
    ),
    "prophet_training_summary": read_csv_optional(
        PROPHET_TRAINING_SUMMARY_PATH, "prophet_training_summary"
    ),
    "prophet_component_summary": read_csv_optional(
        PROPHET_COMPONENT_SUMMARY_PATH, "prophet_component_summary"
    ),
    "final_error_by_date": read_csv_optional(
        FINAL_ERROR_BY_DATE_PATH, "final_error_by_date"
    ),
    "final_error_by_horizon_day": read_csv_optional(
        FINAL_ERROR_BY_HORIZON_PATH, "final_error_by_horizon_day"
    ),
    "final_error_by_store": read_csv_optional(
        FINAL_ERROR_BY_STORE_PATH, "final_error_by_store"
    ),
    "final_error_by_family": read_csv_optional(
        FINAL_ERROR_BY_FAMILY_PATH, "final_error_by_family"
    ),
    "final_error_by_promotion": read_csv_optional(
        FINAL_ERROR_BY_PROMOTION_PATH, "final_error_by_promotion"
    ),
    "final_error_by_holiday": read_csv_optional(
        FINAL_ERROR_BY_HOLIDAY_PATH, "final_error_by_holiday"
    ),
}

# Optional JSON artifacts
lightgbm_experiment_summary = read_json_optional(
    LIGHTGBM_EXPERIMENT_SUMMARY_PATH,
    "lightgbm_experiment_summary",
)
prophet_experiment_summary = read_json_optional(
    PROPHET_EXPERIMENT_SUMMARY_PATH,
    "prophet_experiment_summary",
)

# Infer final model information
if "best_baseline" in baseline_config:
    BEST_BASELINE_MODEL = baseline_config["best_baseline"]
else:
    baseline_name_column = (
        "baseline" if "baseline" in baseline_metrics.columns else "model"
    )
    BEST_BASELINE_MODEL = baseline_metrics.sort_values(["rmsle", "wape", "mae"]).iloc[
        0
    ][baseline_name_column]

recommended_model = final_model_recommendation["recommended_model"].iloc[0]
recommendation_status = final_model_recommendation["recommendation_status"].iloc[0]

loaded_summary = pd.DataFrame(
    [
        {
            "artifact_group": "baseline_metrics",
            "rows": len(baseline_metrics),
            "columns": baseline_metrics.shape[1],
        },
        {
            "artifact_group": "lightgbm_metrics",
            "rows": len(lightgbm_metrics),
            "columns": lightgbm_metrics.shape[1],
        },
        {
            "artifact_group": "prophet_metrics",
            "rows": len(prophet_metrics),
            "columns": prophet_metrics.shape[1],
        },
        {
            "artifact_group": "final_model_leaderboard",
            "rows": len(final_model_leaderboard),
            "columns": final_model_leaderboard.shape[1],
        },
        {
            "artifact_group": "final_model_recommendation",
            "rows": len(final_model_recommendation),
            "columns": final_model_recommendation.shape[1],
        },
    ]
)

display(loaded_summary)
print(f"Best baseline: {BEST_BASELINE_MODEL}")
print(f"Recommended model: {recommended_model}")
print(f"Recommendation status: {recommendation_status}")
print(
    "Section conclusion: existing experiment outputs are loaded and ready for tracking."
)


# 8.Define the MLflow tracking schema

This section documents the intended tracking structure before creating runs.

The schema is saved as a CSV artifact so that the tracking design is part of the project evidence.

### Section conclusion

The MLflow tracking schema defines how models, metrics, parameters and artifacts will be compared over time.


In [ ]:
tracking_schema = pd.DataFrame(
    [
        {
            "run_name": "baseline_*",
            "model_family": "baseline",
            "model_role": "minimum_benchmark",
            "granularity": "date_store_family",
            "main_params": "baseline_name, validation_period, forecast_horizon_days",
            "main_metrics": "rmsle, mae, wape, bias, total_bias_pct",
            "main_artifacts": "baseline metrics, leaderboard, config, predictions, segment tables",
        },
        {
            "run_name": LIGHTGBM_MODEL_VERSION,
            "model_family": "lightgbm",
            "model_role": "champion_candidate",
            "granularity": "date_store_family",
            "main_params": "lightgbm_params, feature list, training controls, validation period",
            "main_metrics": "rmsle, mae, wape, bias, total_bias_pct, train/valid rmse log, best_iteration",
            "main_artifacts": "model txt, config, feature list, metrics, predictions, feature importance, final evaluation",
        },
        {
            "run_name": PROPHET_MODEL_VERSION,
            "model_family": "prophet",
            "model_role": "aggregate_benchmark",
            "granularity": "daily_total_sales",
            "main_params": "prophet params, holidays, validation period",
            "main_metrics": "rmsle, mae, wape, bias, total_bias_pct",
            "main_artifacts": "model json, config, metrics, daily comparison, forecast figures",
        },
        {
            "run_name": "final_model_recommendation",
            "model_family": "selection",
            "model_role": "model_selection_decision",
            "granularity": "project_decision",
            "main_params": "recommended_model, best_baseline, business_use_case, next_step",
            "main_metrics": "recommended model metrics and improvement vs benchmark",
            "main_artifacts": "leaderboard, decision matrix, recommendation, reproducibility notes",
        },
    ]
)

tracking_schema.to_csv(MLFLOW_TRACKING_SCHEMA_PATH, index=False)
display(tracking_schema)

print("Section conclusion: the MLflow tracking schema has been saved.")


# 9.Register baseline experiments

This section registers each baseline as a separate MLflow run.

All baselines are logged under the same experiment so they remain comparable against LightGBM and Prophet. The best baseline receives additional tags and segment artifacts.

### Section conclusion

Baseline runs create the minimum benchmark that future models must beat.


In [ ]:
baseline_run_records = []

baseline_name_column = "baseline" if "baseline" in baseline_metrics.columns else "model"
baseline_metrics_sorted = baseline_metrics.sort_values(
    ["rmsle", "wape", "mae"]
).reset_index(drop=True)

baseline_core_artifacts = [
    BASELINE_METRICS_PATH,
    BASELINE_CONFIG_PATH,
    BASELINE_LEADERBOARD_PATH,
]

baseline_segment_artifacts = [
    BASELINE_METRICS_BY_DATE_PATH,
    BASELINE_METRICS_BY_STORE_PATH,
    BASELINE_METRICS_BY_FAMILY_PATH,
    BASELINE_METRICS_BY_PROMOTION_PATH,
    BASELINE_METRICS_BY_HORIZON_PATH,
]

for _, baseline_row in baseline_metrics_sorted.iterrows():
    baseline_name = str(baseline_row[baseline_name_column])
    is_best_baseline = baseline_name == BEST_BASELINE_MODEL

    with mlflow.start_run(run_name=baseline_name) as run:
        set_common_tags(
            model_family="baseline",
            model_role="best_baseline" if is_best_baseline else "baseline_benchmark",
            model_version=baseline_name,
            granularity="date_store_family",
        )

        mlflow.set_tags(
            {
                "is_best_baseline": str(is_best_baseline).lower(),
                "recommended_for_deployment": "false",
                "supports_store_family_forecast": "true",
            }
        )

        baseline_params = {
            "baseline_name": baseline_name,
            "description": baseline_row.get("description", ""),
            "prediction_grain": "_".join(PREDICTION_GRAIN),
            "target_column": TARGET_COLUMN,
            "forecast_horizon_days": EXPECTED_FORECAST_HORIZON_DAYS,
            "best_baseline": BEST_BASELINE_MODEL,
        }

        baseline_params.update(
            {
                "train_start_date": baseline_config.get("train_start_date"),
                "train_end_date": baseline_config.get("train_end_date"),
                "valid_start_date": baseline_config.get("valid_start_date"),
                "valid_end_date": baseline_config.get("valid_end_date"),
                "leakage_policy": "time_safe_historical_features",
            }
        )

        log_params_safely(baseline_params)
        log_metrics_safely(extract_common_metrics(baseline_row))

        log_artifact_group(baseline_core_artifacts, "baseline/core")

        if is_best_baseline:
            log_artifact_group(
                [BASELINE_VALIDATION_PREDICTIONS_PATH], "baseline/predictions"
            )
            log_artifact_group(baseline_segment_artifacts, "baseline/segment_metrics")

            log_metrics_safely(
                segment_summary_metrics(
                    optional_tables["baseline_metrics_by_date"], "baseline_date"
                )
            )
            log_metrics_safely(
                segment_summary_metrics(
                    optional_tables["baseline_metrics_by_family"], "baseline_family"
                )
            )
            log_metrics_safely(
                segment_summary_metrics(
                    optional_tables["baseline_metrics_by_store"], "baseline_store"
                )
            )

        baseline_run_records.append(
            {
                "run_name": baseline_name,
                "run_id": run.info.run_id,
                "model_family": "baseline",
                "is_best_baseline": is_best_baseline,
                "rmsle": baseline_row.get("rmsle"),
                "wape": baseline_row.get("wape"),
                "mae": baseline_row.get("mae"),
            }
        )

baseline_run_summary = pd.DataFrame(baseline_run_records)
display(baseline_run_summary)

print("Section conclusion: baseline experiments have been registered in MLflow.")


# 10.Register LightGBM v1 experiment

This section registers `lightgbm_v1` as the main operational model.

The run includes:

- model file;
- config;
- feature list;
- validation metrics;
- training diagnostics;
- feature importance;
- predictions and submission artifacts when available;
- final evaluation artifacts from notebook 06.

The model is also logged with the MLflow LightGBM flavor when LightGBM is installed and the saved model can be loaded successfully. If that fails, the raw model file remains registered as an artifact.

### Section conclusion

LightGBM should become the champion candidate for future FastAPI and Docker deployment.


In [ ]:
lightgbm_row = first_row(lightgbm_metrics)
lightgbm_training_row = first_row(optional_tables["lightgbm_training_summary"])
lightgbm_overfitting_row = first_row(optional_tables["lightgbm_overfitting_summary"])

with mlflow.start_run(run_name=LIGHTGBM_MODEL_VERSION) as lightgbm_run:
    set_common_tags(
        model_family="lightgbm",
        model_role="champion",
        model_version=LIGHTGBM_MODEL_VERSION,
        granularity="date_store_family",
    )

    mlflow.set_tags(
        {
            "recommended_for_deployment": "true",
            "deployment_candidate": "true",
            "supports_store_family_forecast": "true",
            "target_transformation": "log1p",
            "negative_prediction_policy": "clip_to_zero",
            "next_phase": "fastapi_docker",
            "business_use_case": "short_term_store_replenishment_and_demand_planning",
        }
    )

    lightgbm_params = {}
    lightgbm_params.update(
        flatten_dict(lightgbm_config.get("lightgbm_params", {}), prefix="lightgbm")
    )
    lightgbm_params.update(
        flatten_dict(lightgbm_config.get("training_controls", {}), prefix="training")
    )
    lightgbm_params.update(
        flatten_dict(lightgbm_config.get("validation_period", {}), prefix="validation")
    )
    lightgbm_params.update(
        {
            "model_version": LIGHTGBM_MODEL_VERSION,
            "model_type": lightgbm_config.get("model_type", "LightGBM regression"),
            "prediction_grain": "_".join(PREDICTION_GRAIN),
            "target_column": TARGET_COLUMN,
            "target_transformation": lightgbm_config.get(
                "target_transformation", "log1p(sales)"
            ),
            "n_features": lightgbm_feature_list.get("n_features"),
            "n_categorical_features": len(
                lightgbm_feature_list.get("categorical_features", [])
            ),
            "n_numeric_features": len(
                lightgbm_feature_list.get("numeric_features", [])
            ),
            "best_baseline": BEST_BASELINE_MODEL,
        }
    )

    log_params_safely(lightgbm_params)
    log_metrics_safely(extract_common_metrics(lightgbm_row))

    # Additional training and overfitting metrics
    log_metrics_safely(
        extract_common_metrics(lightgbm_training_row, include_prefix=False)
    )
    log_metrics_safely(
        extract_common_metrics(lightgbm_overfitting_row, include_prefix=False)
    )

    # Segment summary metrics as compact MLflow scalars
    log_metrics_safely(
        segment_summary_metrics(
            optional_tables["lightgbm_metrics_by_date"], "lightgbm_date"
        )
    )
    log_metrics_safely(
        segment_summary_metrics(
            optional_tables["lightgbm_metrics_by_store"], "lightgbm_store"
        )
    )
    log_metrics_safely(
        segment_summary_metrics(
            optional_tables["lightgbm_metrics_by_family"], "lightgbm_family"
        )
    )
    log_metrics_safely(
        segment_summary_metrics(
            optional_tables["lightgbm_metrics_by_horizon_day"], "lightgbm_horizon"
        )
    )

    # Core artifacts
    log_artifact_group(
        [
            LIGHTGBM_MODEL_PATH,
            LIGHTGBM_CONFIG_PATH,
            LIGHTGBM_FEATURE_LIST_PATH,
            LIGHTGBM_EXPERIMENT_SUMMARY_PATH,
        ],
        "lightgbm/model",
    )

    log_artifact_group(
        [
            LIGHTGBM_METRICS_PATH,
            LIGHTGBM_BASELINE_COMPARISON_PATH,
            LIGHTGBM_VS_BEST_BASELINE_PATH,
            LIGHTGBM_TRAINING_SUMMARY_PATH,
            LIGHTGBM_TRAINING_DIAGNOSTICS_PATH,
            LIGHTGBM_OVERFITTING_SUMMARY_PATH,
            LIGHTGBM_LEARNING_CURVE_PATH,
        ],
        "lightgbm/metrics",
    )

    log_artifact_group(
        [
            LIGHTGBM_FEATURE_IMPORTANCE_PATH,
            LIGHTGBM_FEATURE_GROUP_IMPORTANCE_PATH,
        ],
        "lightgbm/feature_importance",
    )

    log_artifact_group(
        [
            LIGHTGBM_METRICS_BY_DATE_PATH,
            LIGHTGBM_METRICS_BY_STORE_PATH,
            LIGHTGBM_METRICS_BY_FAMILY_PATH,
            LIGHTGBM_METRICS_BY_PROMOTION_PATH,
            LIGHTGBM_METRICS_BY_HORIZON_PATH,
            LIGHTGBM_SEGMENT_COMPARISON_BY_DATE_PATH,
            LIGHTGBM_SEGMENT_COMPARISON_BY_HORIZON_PATH,
            LIGHTGBM_SEGMENT_COMPARISON_BY_STORE_PATH,
            LIGHTGBM_SEGMENT_COMPARISON_BY_FAMILY_PATH,
            LIGHTGBM_SEGMENT_COMPARISON_BY_PROMOTION_PATH,
        ],
        "lightgbm/segment_metrics",
    )

    log_artifact_group(
        [
            LIGHTGBM_VALIDATION_PREDICTIONS_PATH,
            LIGHTGBM_TEST_PREDICTIONS_PATH,
            LIGHTGBM_SUBMISSION_PATH,
        ],
        "lightgbm/predictions",
    )

    log_artifact_group(
        [
            FINAL_MODEL_LEADERBOARD_PATH,
            FINAL_GRANULAR_COMPARISON_PATH,
            FINAL_DAILY_COMPARISON_PATH,
            FINAL_DECISION_MATRIX_PATH,
            FINAL_MODEL_RECOMMENDATION_PATH,
            FINAL_ERROR_BY_DATE_PATH,
            FINAL_ERROR_BY_HORIZON_PATH,
            FINAL_ERROR_BY_STORE_PATH,
            FINAL_ERROR_BY_FAMILY_PATH,
            FINAL_ERROR_BY_PROMOTION_PATH,
            FINAL_ERROR_BY_HOLIDAY_PATH,
        ],
        "lightgbm/final_evaluation",
    )

    log_artifact_group(
        [
            FINAL_GLOBAL_METRICS_FIGURE_PATH,
            FINAL_DAILY_ACTUAL_VS_PREDICTION_FIGURE_PATH,
            FINAL_ERROR_BY_DATE_FIGURE_PATH,
            FINAL_WORST_STORES_FIGURE_PATH,
            FINAL_WORST_FAMILIES_FIGURE_PATH,
        ],
        "lightgbm/final_figures",
    )

    # Optional MLflow LightGBM flavor logging without retraining
    lightgbm_flavor_logged = False

    if LIGHTGBM_AVAILABLE and LIGHTGBM_MODEL_PATH.exists():
        try:
            booster = lgb.Booster(model_file=str(LIGHTGBM_MODEL_PATH))
            mlflow.lightgbm.log_model(
                booster,
                artifact_path="lightgbm/mlflow_model",
            )
            lightgbm_flavor_logged = True
        except Exception as error:
            mlflow.set_tag("lightgbm_flavor_error", str(error)[:500])

    mlflow.set_tag("lightgbm_flavor_logged", str(lightgbm_flavor_logged).lower())

lightgbm_run_id = lightgbm_run.info.run_id
print(f"LightGBM run_id: {lightgbm_run_id}")
print(
    "Section conclusion: LightGBM v1 has been registered as the champion deployment candidate."
)


# 11.Register Prophet aggregate benchmark

This section registers `prophet_total_sales_v1` as an aggregate daily benchmark.

Prophet is intentionally tagged as an interpretable benchmark, not as the final operational model, because it does not produce forecasts at the required `date + store_nbr + family` grain.

### Section conclusion

Prophet remains useful for aggregate diagnostics, trend and seasonality interpretation.


In [ ]:
prophet_row = first_row(prophet_metrics)
prophet_training_row = first_row(optional_tables["prophet_training_summary"])

with mlflow.start_run(run_name=PROPHET_MODEL_VERSION) as prophet_run:
    set_common_tags(
        model_family="prophet",
        model_role="aggregate_benchmark",
        model_version=PROPHET_MODEL_VERSION,
        granularity="daily_total_sales",
    )

    mlflow.set_tags(
        {
            "recommended_for_deployment": "false",
            "deployment_candidate": "false",
            "supports_store_family_forecast": "false",
            "comparison_scope": "daily_aggregate_only",
            "benchmark_against": "lightgbm_v1_daily_aggregate",
        }
    )

    prophet_params = {}
    prophet_params.update(
        flatten_dict(prophet_config.get("prophet_params", {}), prefix="prophet")
    )
    prophet_params.update(
        flatten_dict(prophet_config.get("training_period", {}), prefix="training")
    )
    prophet_params.update(
        flatten_dict(prophet_config.get("validation_period", {}), prefix="validation")
    )
    prophet_params.update(
        flatten_dict(prophet_config.get("holidays", {}), prefix="holidays")
    )
    prophet_params.update(
        {
            "model_version": PROPHET_MODEL_VERSION,
            "model_type": prophet_config.get(
                "model_type", "Prophet aggregate daily time series"
            ),
            "target_grain": prophet_config.get("scope", "daily_total_sales"),
            "target": prophet_config.get("target", "total_daily_sales"),
            "operational_store_family_model": False,
        }
    )

    log_params_safely(prophet_params)
    log_metrics_safely(extract_common_metrics(prophet_row))
    log_metrics_safely(
        extract_common_metrics(prophet_training_row, include_prefix=False)
    )

    log_artifact_group(
        [
            PROPHET_MODEL_PATH,
            PROPHET_CONFIG_PATH,
            PROPHET_EXPERIMENT_SUMMARY_PATH,
        ],
        "prophet/model",
    )

    log_artifact_group(
        [
            PROPHET_METRICS_PATH,
            PROPHET_DAILY_COMPARISON_PATH,
            PROPHET_AGGREGATE_MODEL_COMPARISON_PATH,
            PROPHET_VS_LIGHTGBM_SUMMARY_PATH,
            PROPHET_TRAINING_SUMMARY_PATH,
            PROPHET_COMPONENT_SUMMARY_PATH,
            PROPHET_COMPONENT_DETAIL_PATH,
            PROPHET_WEEKLY_COMPONENT_SUMMARY_PATH,
        ],
        "prophet/metrics",
    )

    log_artifact_group(
        [
            PROPHET_VALIDATION_PREDICTIONS_PATH,
        ],
        "prophet/predictions",
    )

    log_artifact_group(
        [
            PROPHET_FORECAST_FIGURE_PATH,
            PROPHET_COMPONENTS_FIGURE_PATH,
        ],
        "prophet/figures",
    )

prophet_run_id = prophet_run.info.run_id
print(f"Prophet run_id: {prophet_run_id}")
print(
    "Section conclusion: Prophet has been registered as an aggregate benchmark, not as the deployment model."
)


# 12.Register final model recommendation

This section creates a dedicated MLflow run for the model selection decision.

This run does not represent a trained model. It represents the architecture and evaluation decision that connects experimentation with the next MLOps phase.

### Section conclusion

The final recommendation should make the deployment candidate explicit and auditable.


In [ ]:
recommendation_row = first_row(final_model_recommendation)
decision_params = recommendation_row.to_dict()

with mlflow.start_run(run_name="final_model_recommendation") as final_decision_run:
    set_common_tags(
        model_family="selection",
        model_role="model_selection_decision",
        model_version=str(recommended_model),
        granularity="project_decision",
    )

    mlflow.set_tags(
        {
            "recommended_model": str(recommended_model),
            "recommendation_status": str(recommendation_status),
            "best_baseline": str(BEST_BASELINE_MODEL),
            "prophet_role": "aggregate_interpretable_benchmark",
            "next_step": "prepare_fastapi_contract",
            "deployment_candidate": str(
                recommended_model == LIGHTGBM_MODEL_VERSION
            ).lower(),
        }
    )

    log_params_safely(
        {
            "recommended_model": recommendation_row.get("recommended_model"),
            "model_version": recommendation_row.get("model_version"),
            "recommendation_status": recommendation_row.get("recommendation_status"),
            "recommended_use": recommendation_row.get("recommended_use"),
            "business_use_case": recommendation_row.get("business_use_case"),
            "validation_grain": recommendation_row.get("validation_grain"),
            "forecast_horizon_days": recommendation_row.get("forecast_horizon_days"),
            "best_baseline": recommendation_row.get(
                "best_baseline", BEST_BASELINE_MODEL
            ),
            "prophet_role": recommendation_row.get("prophet_role"),
            "next_step": recommendation_row.get("next_step"),
        }
    )

    log_metrics_safely(extract_common_metrics(recommendation_row))
    log_metrics_safely(
        {
            "best_baseline_rmsle": recommendation_row.get("best_baseline_rmsle"),
            "best_baseline_wape": recommendation_row.get("best_baseline_wape"),
        }
    )

    log_artifact_group(
        [
            FINAL_MODEL_LEADERBOARD_PATH,
            FINAL_GRANULAR_COMPARISON_PATH,
            FINAL_DAILY_COMPARISON_PATH,
            FINAL_DECISION_MATRIX_PATH,
            FINAL_MODEL_RECOMMENDATION_PATH,
            MLFLOW_ARTIFACTS_RECOMMENDATION_PATH,
            MLFLOW_TRACKING_SCHEMA_PATH,
            MLFLOW_ARTIFACT_INVENTORY_PATH,
        ],
        "final_decision",
    )

final_decision_run_id = final_decision_run.info.run_id
print(f"Final recommendation run_id: {final_decision_run_id}")
print(f"Recommended model: {recommended_model}")
print(
    "Section conclusion: the final model recommendation has been registered in MLflow."
)


# 13.Save MLflow artifact audit

This section saves an audit table of all artifacts logged during this notebook.

The table is useful for reviewing what was registered without opening the MLflow UI.

### Section conclusion

Artifact logging is documented outside MLflow as an additional local control table.


In [ ]:
registered_artifacts = pd.DataFrame(logged_artifact_records)

if registered_artifacts.empty:
    registered_artifacts = pd.DataFrame(
        columns=[
            "run_id",
            "run_name",
            "artifact_path",
            "source_path",
            "relative_source_path",
            "file_size_mb",
            "logged_at",
        ]
    )

registered_artifacts.to_csv(MLFLOW_REGISTERED_ARTIFACTS_PATH, index=False)
display(registered_artifacts.head(30))

print(f"Logged artifact files: {len(registered_artifacts):,}")
print(
    "Section conclusion: registered artifacts have been saved as a local audit table."
)


# 14.MLflow run summary and validation

This section queries MLflow and creates a compact summary of the runs registered by this notebook.

The summary table is saved in `reports/tables/` and can be used in the README or portfolio documentation.

### Section conclusion

The registered runs should make baseline, LightGBM, Prophet and final recommendation comparable and auditable.


In [ ]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

if experiment is None:
    raise ValueError(f"MLflow experiment not found: {MLFLOW_EXPERIMENT_NAME}")

mlflow_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    output_format="pandas",
)

run_name_column = "tags.mlflow.runName"

selected_summary_columns = [
    "run_id",
    "status",
    "start_time",
    "end_time",
    run_name_column,
    "tags.model_family",
    "tags.model_role",
    "tags.model_version",
    "tags.granularity",
    "tags.recommended_for_deployment",
    "metrics.valid_rmsle",
    "metrics.valid_mae",
    "metrics.valid_wape",
    "metrics.valid_bias",
    "metrics.valid_total_bias_pct",
]

available_summary_columns = [
    column for column in selected_summary_columns if column in mlflow_runs.columns
]

mlflow_run_summary = (
    mlflow_runs[available_summary_columns]
    .copy()
    .sort_values("start_time", ascending=False)
    .reset_index(drop=True)
)

mlflow_run_summary.to_csv(MLFLOW_RUN_SUMMARY_PATH, index=False)

display(mlflow_run_summary)

expected_run_names = set(
    baseline_metrics_sorted[baseline_name_column].astype(str).tolist()
)
expected_run_names.update(
    {
        LIGHTGBM_MODEL_VERSION,
        PROPHET_MODEL_VERSION,
        "final_model_recommendation",
    }
)

registered_run_names = set(mlflow_runs[run_name_column].dropna().astype(str).tolist())

missing_expected_runs = sorted(expected_run_names - registered_run_names)

if missing_expected_runs:
    raise ValueError(f"Missing expected MLflow runs: {missing_expected_runs}")

print(f"Registered expected runs: {len(expected_run_names)}")
print(
    "Section conclusion: MLflow contains all expected runs for this tracking notebook."
)


# 15.Reproducibility notes and limitations

This section documents what is reproducible from this notebook and what remains outside the current MLOps scope.

These notes are saved as a table so they can be reused in the README and future deployment documentation.

### Section conclusion

The project has local experiment traceability, but not yet production-grade model governance.


In [ ]:
reproducibility_notes = pd.DataFrame(
    [
        {
            "category": "reproducibility",
            "item": "local_tracking",
            "note": "MLflow tracking is stored locally under the project mlruns directory.",
        },
        {
            "category": "reproducibility",
            "item": "no_retraining",
            "note": "This notebook registers existing artifacts and does not retrain models.",
        },
        {
            "category": "reproducibility",
            "item": "artifact_dependency",
            "note": "Runs depend on artifacts produced by notebooks 03, 04, 05 and 06.",
        },
        {
            "category": "reproducibility",
            "item": "validation_window",
            "note": "Model comparison is based on a 16-day temporal holdout validation period.",
        },
        {
            "category": "reproducibility",
            "item": "operational_grain",
            "note": "The deployment candidate must forecast at date, store_nbr and family grain.",
        },
        {
            "category": "limitation",
            "item": "single_validation_window",
            "note": "The final decision is based on a single validation window, not repeated rolling validation.",
        },
        {
            "category": "limitation",
            "item": "prophet_granularity",
            "note": "Prophet is only comparable at daily total sales level and is not an operational store-family model.",
        },
        {
            "category": "limitation",
            "item": "no_dataset_versioning",
            "note": "The project does not yet use DVC or a formal dataset registry.",
        },
        {
            "category": "limitation",
            "item": "no_remote_registry",
            "note": "The project uses local MLflow tracking and does not use a remote model registry.",
        },
        {
            "category": "next_step",
            "item": "api_contract",
            "note": "The next notebook or project phase should define the FastAPI input schema for LightGBM inference.",
        },
    ]
)

reproducibility_notes.to_csv(MLFLOW_REPRODUCIBILITY_NOTES_PATH, index=False)
display(reproducibility_notes)

log_artifact_if_exists(MLFLOW_REPRODUCIBILITY_NOTES_PATH, "project_documentation")
log_artifact_if_exists(MLFLOW_RUN_SUMMARY_PATH, "project_documentation")
log_artifact_if_exists(MLFLOW_REGISTERED_ARTIFACTS_PATH, "project_documentation")

print("Section conclusion: reproducibility notes and limitations have been documented.")


# 16.Notebook closure criteria

This section validates whether the MLflow tracking notebook can be considered complete.

The notebook is complete only if:

- required artifacts exist;
- baseline runs are registered;
- the LightGBM run is registered;
- the Prophet run is registered;
- the final recommendation run is registered;
- the recommended model is `lightgbm_v1`;
- run summaries and audit tables were saved;
- reproducibility notes were saved.

### Section conclusion

The tracking notebook should close only when the experiment evidence is complete and reusable.


In [ ]:
closure_checks = pd.DataFrame(
    [
        {
            "check": "required_artifacts_available",
            "passed": missing_required_artifacts.empty,
            "details": "All required input artifacts exist.",
        },
        {
            "check": "baseline_runs_registered",
            "passed": set(
                baseline_metrics_sorted[baseline_name_column].astype(str)
            ).issubset(registered_run_names),
            "details": "Each baseline has an MLflow run.",
        },
        {
            "check": "lightgbm_run_registered",
            "passed": LIGHTGBM_MODEL_VERSION in registered_run_names,
            "details": "LightGBM v1 has an MLflow run.",
        },
        {
            "check": "prophet_run_registered",
            "passed": PROPHET_MODEL_VERSION in registered_run_names,
            "details": "Prophet aggregate benchmark has an MLflow run.",
        },
        {
            "check": "final_recommendation_run_registered",
            "passed": "final_model_recommendation" in registered_run_names,
            "details": "The final model decision has an MLflow run.",
        },
        {
            "check": "recommended_model_is_lightgbm",
            "passed": str(recommended_model) == LIGHTGBM_MODEL_VERSION,
            "details": f"Recommended model: {recommended_model}",
        },
        {
            "check": "mlflow_run_summary_saved",
            "passed": MLFLOW_RUN_SUMMARY_PATH.exists(),
            "details": relative_to_project(MLFLOW_RUN_SUMMARY_PATH),
        },
        {
            "check": "registered_artifact_audit_saved",
            "passed": MLFLOW_REGISTERED_ARTIFACTS_PATH.exists(),
            "details": relative_to_project(MLFLOW_REGISTERED_ARTIFACTS_PATH),
        },
        {
            "check": "reproducibility_notes_saved",
            "passed": MLFLOW_REPRODUCIBILITY_NOTES_PATH.exists(),
            "details": relative_to_project(MLFLOW_REPRODUCIBILITY_NOTES_PATH),
        },
    ]
)

closure_checks.to_csv(MLFLOW_NOTEBOOK_CLOSURE_PATH, index=False)
display(closure_checks)

if not closure_checks["passed"].all():
    failed_checks = closure_checks.query("passed == False")
    raise ValueError(
        "MLflow tracking notebook is not complete:\n"
        + "\n".join(
            [
                f"- {row.check}: {row.details}"
                for row in failed_checks.itertuples(index=False)
            ]
        )
    )

final_tracking_summary = pd.DataFrame(
    [
        {
            "project": "store_sales_forecasting",
            "mlflow_experiment_name": MLFLOW_EXPERIMENT_NAME,
            "mlflow_tracking_uri": MLFLOW_TRACKING_URI,
            "recommended_model": recommended_model,
            "recommendation_status": recommendation_status,
            "best_baseline": BEST_BASELINE_MODEL,
            "prophet_role": "aggregate_interpretable_benchmark",
            "next_step": "prepare_fastapi_and_docker_serving_layer",
            "closed_at": datetime.now().isoformat(timespec="seconds"),
        }
    ]
)

final_tracking_summary.to_csv(MLFLOW_FINAL_MODEL_TRACKING_SUMMARY_PATH, index=False)
display(final_tracking_summary)

print("Section conclusion: all closure criteria passed.")


# Final conclusion

The MLflow tracking layer has been created successfully.

The project now has local experiment tracking for:

- baseline benchmarks;
- `lightgbm_v1`;
- `prophet_total_sales_v1`;
- the final model recommendation.

`lightgbm_v1` is tracked as the final recommended model for the next MLOps phase because it supports the required `date + store_nbr + family` forecasting grain and is the best candidate for future FastAPI and Docker deployment.

Prophet remains registered as an aggregate interpretable benchmark, not as the final operational model.
